# BP3 Gate 5 — Decision Layer & Reporting
**Customer360 Navigator Enterprise Suite — Complaint Escalation / Intervention Prediction**

## Purpose
Turns Gate 4's validated champion into a per-row decision-record layer: a prediction, a
confidence score, and a grounded reason code for every real held-out test row — the same
governance step BP1/BP2 Gate 5 built, reused here (HYPER) with BP3's binary
intervention-required target in place of BP1's 77-class intent / BP2's 4-class severity.

## Scope decision carried over from BP1/BP2 (same standing decision, not re-litigated here)
This gate is a fully offline, deterministic decision-record layer — **no GenAI API call**. Real
GenAI-drafted customer-facing text (and the UDAAP/NIST AI RMF review that requires) is scoped to
BP6 per the Master Execution Plan, the identical scope decision the user confirmed for BP1 and
BP2 Gate 5.

## Real Gate 3/4 results this gate builds on
Gate 3 champion = `xgboost` (held-out test PR-AUC=0.3496, ROC-AUC=0.9762, recall=0.9424 at the
default 0.5 threshold). Gate 4 confirmed this champion statistically (near-zero drift on its own
CV re-run, consistency_check_diff=3e-06), added a bootstrap-CI'd PR-AUC/ROC-AUC, a 9-point
threshold analysis (F1-maximizing threshold in that grid = 0.9), quantile-binned calibration, and
— critically — a disparate-impact monitoring check on the barred `Tags` column (read-only,
ECOA/Reg B) that came back **flagged**: adverse-impact ratio = 0.139 (well below the 0.8
four-fifths-rule convention), driven by a real, much higher positive-intervention rate in the
Older-American/Older-American+Servicemember subgroups (~10.3%) versus untagged rows (~1.06%).
Both champion and this flagged finding are read **live** — champion from
`gate4_statistical_validation.json`, cross-checked against Gate 3's config block; the
disparate-impact number is independently *recomputed* here (not copied) on this gate's own full
refit + full held-out test set, then cross-checked against Gate 4's recorded value.

## Design decision 1: binary target replaces BP1/BP2's top-3-confidence breakdown
BP1 (77 intents) and BP2 (4 severities) both reported a ranked top-3 (or top-1/2/3-of-4)
confidence breakdown per row. BP3's target is binary (`intervention_required` ∈ {0, 1}) — there
is no second- or third-ranked class to report; `predict_proba` has exactly one free parameter
(the positive-class probability; the negative-class probability is `1 - p` by construction, not
new information). Each decision record therefore carries:
- `predicted_label` — 0/1 at the **default 0.5 threshold**, matching Gate 3's official
  primary-reporting threshold (never silently switched).
- `predicted_probability` — the raw positive-class probability, reported directly as the
  confidence score (no rounding into a "top-1 of 2" framing that would add nothing).
- `predicted_label_at_best_f1_threshold` — the same probability re-thresholded at Gate 4's
  documented alternate F1-maximizing threshold (0.9, from its 9-point grid), carried through
  **explicitly labeled as an alternate reference point**, never replacing the primary 0.5-based
  label. At threshold 0.9 almost no rows are predicted positive — Gate 4 recorded that number
  honestly as a directional F1-optimizing result on a 9-point grid, not a recommended
  operating point, and this gate repeats that framing rather than quietly adopting 0.9 as primary.

## Design decision 2: the real, flagged Gate 4 adverse-impact finding is carried forward, not silently dropped
Master Plan Section 8 frames Gate 5 as producing "a decision-engine score with reason codes"
under the same compliance touchpoint that named the ECOA/Reg B disparate-impact check at Gate 4.
A governance-focused decision layer that silently omits an already-confirmed, flagged fairness
finding would itself be a gap. This gate re-derives `Tags` as the identical read-only passthrough
column Gate 4 used (never one-hot encoded, never fit on, still asserted absent from the actual
model feature frame), recomputes the same per-group selection-rate/adverse-impact-ratio check on
its own full refit + full held-out test set (not the same code copy-pasted blindly — Gate 5 must
reach the same number as Gate 4 on the identical split, which is itself a live consistency check
that the two gates' feature engineering has not drifted apart), and:
- Adds a `tags_group` column to every decision record (real value carried through, never a
  feature) so a downstream reviewer can filter decision records by group.
- Surfaces the recomputed adverse-impact ratio and its four-fifths-rule flag prominently in this
  gate's own summary JSON and printed output — not buried, and not just a re-print of Gate 4's
  number: an independently recomputed value cross-checked against Gate 4's for consistency.
- States plainly, again, that this is a monitoring signal for a human reviewer, not a legal
  determination — the same limitation language Gate 4 used, not softened here.

## What this notebook does
1. Reads the real champion **live** from `gate4_statistical_validation.json`, cross-checks it
   against Gate 3's recorded champion — never hardcoded here. Also reads Gate 4's recorded
   `best_f1_threshold_in_grid` and `adverse_impact_ratio_tags` live, for the two cross-checks
   above.
2. Rebuilds Gate 3/4's exact feature engineering via the unmodified
   `features.bp3_escalation_features` module (`BARRED_COLUMNS`, `COMPANY_COL`,
   `FEATURE_COLS_CATEGORICAL`) plus `Tags` as a read-only passthrough column, and refits the
   champion on the full real train split (identical `train_test_split` call as Gates 3/4).
3. Predicts on the full real held-out test set (not a sample) — every row gets a decision record.
4. Recomputes held-out test PR-AUC (average_precision — BP3's champion-selection metric, never
   accuracy, which BP3 never computes anywhere) and cross-checks it against Gate 3's recorded
   `held_out_test_pr_auc`.
5. Recomputes the disparate-impact check (Design decision 2 above) and cross-checks it against
   Gate 4's recorded `adverse_impact_ratio_tags`.
6. Computes per-instance SHAP on a bounded 150-row sample (laptop-safe), producing grounded
   reason codes: for the shared one-hot/frequency feature space, a reason code is only ever drawn
   from a feature that is literally nonzero in that row (the category genuinely present in that
   complaint's own data) — same masking principle BP1/BP2 Gate 5 used. No raw-categorical
   (`Column=Value`) path is needed — BP3 never includes CatBoost as a candidate (Lesson #22).
7. Cross-checks its own sample-aggregated top reason-code terms against Gate 4's saved global
   top-10 SHAP features (`gate4_shap_top_features.csv`).

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: Claude wrote this notebook; it does not run it.
  Every number — predictions, probabilities, the recomputed PR-AUC, the recomputed
  disparate-impact ratio, SHAP reason codes — is computed live during the real run.
- **HYPER**: Gate 3/4's feature-engineering code (Gold-layer reload via the shared
  `features.bp3_escalation_features` module, train/test split, shared one-hot/frequency
  preprocessing) is reused verbatim so this gate's refit/predict is meaningful only on the
  IDENTICAL rows/columns Gates 3/4 used — required for both consistency checks to mean anything,
  not just convenient. `src/utils/bp1_config_sync.py` reused unmodified.
- **Continue gracefully on failure**: SHAP computation is wrapped in try/except — a failure is
  recorded plainly (`shap_error`) and decision records still get written with empty reason codes
  rather than the whole gate halting.
- **Idempotent**: re-running overwrites this gate's artifacts and its own `gate5_...` config
  block, without touching Gates 1-4's fields.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate5_decision_records.csv` (one row
  per real held-out test example — includes `tags_group` per Design decision 2)
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate5_decision_layer_summary.json`
  (includes the recomputed disparate-impact ratio, its consistency check against Gate 4, and the
  compliance touchpoint)
- `notebooks/bp3_complaint_escalation_prediction/artifacts/model_inventory_entry.json` (Gate 5
  fields added to the existing Gate 3/4 entry, in place)
- `configs/bp3_complaint_escalation_prediction.yaml` — `gate5_decision_layer` block
  appended/updated

## Prerequisites
BP3 Gates 3 and 4 must both have been real-run (this notebook reads Gate 4's confirmed champion,
best-F1 threshold, and adverse-impact ratio, and raises if `gate4_statistical_validation.json` is
missing). `shap` must be installed — same live check Gate 4 performs.

## If a structural check below fails
It raises `AssertionError` naming the failing check. If the PR-AUC-consistency check fails, this
notebook's feature-engineering code has drifted from Gate 3/4's — fix the drift, do not silence
it. If the disparate-impact-consistency check fails, the same applies to the `Tags` passthrough
logic specifically.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Gate 5 decision layer / reporting notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os
import sys
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.util  # noqa: E402
import re  # noqa: E402
import time  # noqa: E402
import yaml  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
from scipy import sparse as sp  # noqa: E402
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import average_precision_score, recall_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402
from sklearn.preprocessing import OneHotEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402

from features.bp3_escalation_features import (  # noqa: E402
    BARRED_COLUMNS,
    COMPANY_COL,
    FEATURE_COLS_CATEGORICAL,
)

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

if importlib.util.find_spec("shap") is None:
    raise ImportError(
        "[CHECK FAILED] The 'shap' package is required for BP3 Gate 5 and was not confirmed "
        "installed. Run `pip install shap` (inside this project's own environment) before "
        "running this notebook."
    )
import shap  # noqa: E402

print(f"[OK] shap {shap.__version__} confirmed installed (live check, not assumed).")

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

GOLD_PATH = DATA_PROCESSED / "cfpb_intervention_escalation_gold.parquet"
BP3_CONFIG_PATH = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"

for p in (GOLD_PATH, BP3_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP3 Gates 1-4 all completed for real."
        )

# ============================================================
# SECTION 4: Load Gate 3/4's real results - champion, alternate threshold, and the flagged
# disparate-impact ratio all read LIVE, never hardcoded here.
# ============================================================
with open(BP3_CONFIG_PATH, "r", encoding="utf-8") as f:
    bp3_config = yaml.safe_load(f)

TARGET_COL = bp3_config["target_definition"]["primary_target"]
RANDOM_STATE = bp3_config["random_state"]

gate3_block = bp3_config.get("gate3_model_benchmark")
assert gate3_block is not None, (
    "[CHECK FAILED] gate3_model_benchmark is missing from "
    "configs/bp3_complaint_escalation_prediction.yaml - run BP3 Gate 3 first."
)

gate4_json_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
assert gate4_json_path.exists(), (
    f"[CHECK FAILED] {gate4_json_path} not found - run BP3 Gate 4 first (this gate reads its "
    "confirmed champion, alternate threshold, and disparate-impact ratio)."
)
with open(gate4_json_path, "r", encoding="utf-8") as f:
    gate4_results = json.load(f)

CHAMPION_NAME = gate4_results["champion_model"]
assert CHAMPION_NAME == gate3_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: Gate 4 recorded '{CHAMPION_NAME}' but Gate 3's config "
    f"block says '{gate3_block['champion_model']}' - these must agree; re-run Gate 3/4."
)
gate3_recorded_test_pr_auc = float(gate3_block["held_out_test_pr_auc"])
gate4_best_f1_threshold = float(gate4_results["best_f1_threshold_in_grid"])
gate4_recorded_adverse_impact_ratio = gate4_results.get("adverse_impact_ratio_tags")
print(f"[OK] Champion (live, re-verified against Gate 3 + Gate 4): {CHAMPION_NAME}")
print(f"[OK] Gate 4's alternate F1-maximizing threshold (live, reference only): {gate4_best_f1_threshold}")
print(
    f"[OK] Gate 4's recorded disparate-impact ratio (Tags, live, for the consistency check "
    f"below): {gate4_recorded_adverse_impact_ratio}"
)

cv_settings = RESOURCE_LIMITS["cv"]
RNG = np.random.RandomState(cv_settings["random_state"])

# ============================================================
# SECTION 5: Rebuild the real Gold-layer feature frame EXACTLY as Gates 3/4 did (HYPER reuse via
# the unmodified Gate 2 module) - this gate's refit/predict and both consistency checks are only
# meaningful on the IDENTICAL rows/columns Gates 3/4 used. 'Tags' is carried through as a
# read-only passthrough column (never a feature) for the disparate-impact recomputation in
# Section 10, same as Gate 4.
# ============================================================
gold_lazy = pl.scan_parquet(GOLD_PATH)
select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL, "Tags"]
df_pl = gold_lazy.select(select_cols).filter(pl.col(TARGET_COL).is_not_null()).collect()
for barred in BARRED_COLUMNS:
    assert barred not in FEATURE_COLS_CATEGORICAL + [
        COMPANY_COL
    ], f"[CHECK FAILED] barred column '{barred}' present in the modeling feature list."
print(f"[OK] Reloaded real Gold layer, trainable rows: {df_pl.height:,} (must match Gate 3/4's row count).")

feature_data = {col: df_pl[col].cast(pl.Utf8).to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
feature_data["Tags"] = df_pl["Tags"].cast(pl.Utf8).fill_null("NO_TAG").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Int8).to_list()
X_full = pd.DataFrame(feature_data)
y_full = pd.Series(target_data, name=TARGET_COL)
print(
    f"[OK] Built feature+aux frame: {X_full.shape[0]:,} rows x {X_full.shape[1]} columns "
    f"({FEATURE_COLS_CATEGORICAL + [COMPANY_COL]} as model features, 'Tags' as a read-only "
    "passthrough column for Section 10 only)."
)

# ============================================================
# SECTION 6: Identical stratified train/test split as Gates 3/4
# ============================================================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.20, stratify=y_full, random_state=RANDOM_STATE
)
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()
print(f"[OK] Reproduced Gate 3/4's train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,}.")

n_train_positive = int((y_train == 1).sum())
n_train_negative = int((y_train == 0).sum())
scale_pos_weight = n_train_negative / n_train_positive
print(
    f"[OK] Reproduced Gate 3/4's live scale_pos_weight={scale_pos_weight:.2f} from the identical train split."
)

# ============================================================
# SECTION 7: Rebuild the shared preprocessing EXACTLY as Gates 3/4 (fit on TRAIN only). 'Tags' is
# explicitly excluded from this ColumnTransformer - it is never one-hot encoded, never seen by
# any model.
# ============================================================
ohe = ColumnTransformer(
    [("ohe", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), FEATURE_COLS_CATEGORICAL)],
    remainder="drop",
)
X_train_ohe = ohe.fit_transform(X_train_raw)
X_test_ohe = ohe.transform(X_test_raw)

company_freq_map = X_train_raw[COMPANY_COL].value_counts().to_dict()
train_company_freq = (
    X_train_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
)
test_company_freq = (
    X_test_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
)

X_train_shared = sp.hstack([X_train_ohe, sp.csr_matrix(train_company_freq)], format="csr")
X_test_shared = sp.hstack([X_test_ohe, sp.csr_matrix(test_company_freq)], format="csr")
SHARED_FEATURE_NAMES = np.array(list(ohe.get_feature_names_out()) + ["Company_freq"])
print(
    f"[OK] Reproduced Gate 3/4's shared feature matrix: train={X_train_shared.shape}, "
    f"test={X_test_shared.shape}."
)

# ============================================================
# SECTION 8: Candidate definition - MUST exactly mirror Gate 3/4's (single-source-of-truth risk,
# guarded by the PR-AUC consistency check in Section 9 - if this notebook's definition ever drifts
# from Gate 3/4's, that check catches it).
# ============================================================
CANDIDATES = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=cv_settings["random_state"]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight="balanced",
        n_jobs=1,
        random_state=cv_settings["random_state"],
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    "xgboost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        n_jobs=1,
        verbosity=0,
        scale_pos_weight=scale_pos_weight,
        random_state=cv_settings["random_state"],
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100,
        class_weight="balanced",
        n_jobs=1,
        verbose=-1,
        random_state=cv_settings["random_state"],
    ),
}
NEEDS_DENSE = {"hist_gradient_boosting"}

# ============================================================
# SECTION 9: Refit champion on FULL train, predict on the FULL held-out test set. Recompute
# held-out test PR-AUC (average_precision - BP3's champion-selection metric, never accuracy) and
# cross-check it against Gate 3's recorded value.
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
X_train_final, X_test_final = X_train_shared, X_test_shared
if CHAMPION_NAME in NEEDS_DENSE:
    X_train_final = np.asarray(X_train_final.todense(), dtype=np.float32)
    X_test_final = np.asarray(X_test_final.todense(), dtype=np.float32)

champion_model = CANDIDATES[CHAMPION_NAME]
print(f"\n[GATE5] Refitting champion ({CHAMPION_NAME}) on the full train split...")
t0 = time.perf_counter()
champion_model.fit(X_train_final, y_train)
print(f"[GATE5] Fit done in {time.perf_counter() - t0:.1f}s")

assert hasattr(champion_model, "predict_proba"), (
    f"[CHECK FAILED] Champion {CHAMPION_NAME} has no predict_proba - Gate 5's confidence score "
    "requires it."
)
y_proba = champion_model.predict_proba(X_test_final)[:, 1]
proba_valid_range = bool(np.all((y_proba >= 0.0) & (y_proba <= 1.0)))
assert proba_valid_range, "[CHECK FAILED] predict_proba positive-class values fall outside [0, 1]."

# Binary target - no top-2/top-3 ranking exists (the negative-class probability is 1 - p, not new
# information). predicted_label is reported at the default 0.5 threshold (Gate 3's official
# primary-reporting threshold, never silently switched); predicted_label_at_best_f1_threshold is
# carried alongside as an explicitly labeled alternate reference point from Gate 4's 9-point grid.
y_pred_default = (y_proba >= 0.5).astype(int)
y_pred_alt_threshold = (y_proba >= gate4_best_f1_threshold).astype(int)

recomputed_test_pr_auc = float(average_precision_score(y_test, y_proba))
pr_auc_consistency_diff = abs(recomputed_test_pr_auc - gate3_recorded_test_pr_auc)
print(
    f"[CHECK] Recomputed held-out test PR-AUC: {recomputed_test_pr_auc:.6f} "
    f"(Gate 3 recorded: {gate3_recorded_test_pr_auc:.6f}, diff={pr_auc_consistency_diff:.6f})"
)

mean_conf_correct = (
    float(y_proba[y_pred_default == y_test].mean()) if (y_pred_default == y_test).any() else None
)
mean_conf_incorrect = (
    float(y_proba[y_pred_default != y_test].mean()) if (y_pred_default != y_test).any() else None
)
print(
    f"[RESULT] Mean predicted probability - correct predictions: "
    f"{round(mean_conf_correct, 4) if mean_conf_correct is not None else None}, incorrect "
    f"predictions: {round(mean_conf_incorrect, 4) if mean_conf_incorrect is not None else None}"
)
print(
    f"[RESULT] Predicted-positive counts - default 0.5 threshold: {int(y_pred_default.sum()):,}, "
    f"alternate F1-maximizing threshold ({gate4_best_f1_threshold}, Gate 4 reference only): "
    f"{int(y_pred_alt_threshold.sum()):,}."
)

# ============================================================
# SECTION 10: Disparate-impact monitoring check (ECOA/Reg B) - independently RECOMPUTED here on
# this gate's own full refit + full held-out test set (never copied from Gate 4), then
# cross-checked against Gate 4's recorded value. 'Tags' stays read-only throughout - never a
# feature, never fit on.
# ============================================================
tags_test = X_test_raw["Tags"].to_numpy()
disparate_rows = []
for tag_value in ["NO_TAG", "Servicemember", "Older American", "Older American, Servicemember"]:
    mask = tags_test == tag_value
    n_group = int(mask.sum())
    if n_group == 0:
        continue
    group_pred = y_pred_default[mask]
    group_actual = y_test[mask]
    selection_rate = float(group_pred.mean())
    group_recall = (
        float(recall_score(group_actual, group_pred, zero_division=0)) if group_actual.sum() > 0 else None
    )
    disparate_rows.append(
        {
            "tags_group": tag_value,
            "n_rows_in_test": n_group,
            "n_real_positive_in_group": int(group_actual.sum()),
            "selection_rate_at_0.5_threshold": round(selection_rate, 4),
            "recall_at_0.5_threshold": round(group_recall, 4) if group_recall is not None else None,
        }
    )
disparate_df = pd.DataFrame(disparate_rows)
selection_rates = disparate_df["selection_rate_at_0.5_threshold"]
recomputed_adverse_impact_ratio = (
    float(selection_rates.min() / selection_rates.max()) if selection_rates.max() > 0 else None
)
adverse_impact_consistency_diff = (
    abs(recomputed_adverse_impact_ratio - float(gate4_recorded_adverse_impact_ratio))
    if (recomputed_adverse_impact_ratio is not None and gate4_recorded_adverse_impact_ratio is not None)
    else None
)
print("\n=== DISPARATE-IMPACT MONITORING CHECK (Tags, ECOA/Reg B - read-only, recomputed) ===")
print(disparate_df.to_string(index=False))
if recomputed_adverse_impact_ratio is not None:
    flag = (
        "FLAGGED (< 0.8, four-fifths-rule convention)"
        if recomputed_adverse_impact_ratio < 0.8
        else "within four-fifths-rule convention"
    )
    print(
        f"[RESULT] Recomputed adverse-impact ratio (min/max group selection rate): "
        f"{recomputed_adverse_impact_ratio:.4f} - {flag}."
    )
    diff_display = (
        round(adverse_impact_consistency_diff, 6) if adverse_impact_consistency_diff is not None else None
    )
    print(
        f"[CHECK] Cross-check against Gate 4's recorded value "
        f"({gate4_recorded_adverse_impact_ratio}): diff={diff_display}"
    )
    print(
        "[LIMITATION] This is a monitoring signal for a human reviewer, not a legal "
        "determination of ECOA/Reg B compliance - carried forward from Gate 4, not re-litigated "
        "or softened here."
    )

# ============================================================
# SECTION 11: Per-instance SHAP - explainer chosen by the champion's actual model type, bounded
# sample for laptop safety, reason codes grounded by construction: only nonzero-valued features in
# that row are ever reported (the categories literally present in that complaint's own data). No
# raw-categorical path needed - BP3 never includes CatBoost as a candidate (Lesson #22).
# ============================================================
SHAP_SAMPLE_SIZE = min(150, len(X_test_raw))
SHAP_BACKGROUND_SIZE = min(50, len(X_train_raw))
N_REASON_CODES = 5
shap_error = None
sample_idx = np.array([], dtype=int)
reason_codes_by_row = {}
grounding_failures = 0
is_linear_champion = isinstance(champion_model, LogisticRegression)
try:
    shap_rng = np.random.RandomState(cv_settings["random_state"])
    sample_idx = shap_rng.choice(len(X_test_raw), size=SHAP_SAMPLE_SIZE, replace=False)
    bg_idx = shap_rng.choice(len(X_train_raw), size=SHAP_BACKGROUND_SIZE, replace=False)

    X_sample_vec = X_test_shared[sample_idx]
    X_bg_vec = X_train_shared[bg_idx]
    feature_names = SHARED_FEATURE_NAMES
    X_sample_dense_for_masking = np.asarray(X_sample_vec.todense())
    if not is_linear_champion:
        X_sample_vec = np.asarray(X_sample_vec.todense(), dtype=np.float32)
        X_bg_vec = np.asarray(X_bg_vec.todense(), dtype=np.float32)
    X_sample, X_bg = X_sample_vec, X_bg_vec

    if is_linear_champion:
        print(
            f"\n[GATE5] SHAP: using LinearExplainer for {CHAMPION_NAME} "
            f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)..."
        )
        explainer = shap.LinearExplainer(champion_model, X_bg)
        shap_values = explainer.shap_values(X_sample)
    else:
        print(
            f"\n[GATE5] SHAP: using TreeExplainer for {CHAMPION_NAME} "
            f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)..."
        )
        explainer = shap.TreeExplainer(champion_model)
        shap_values = explainer.shap_values(X_sample)

    # Binary classifier: SHAP values are for the positive class (or a 2-class list/array); every
    # decision record's reason codes explain the SAME positive-class score the record reports.
    if isinstance(shap_values, list):
        per_row_shap = np.asarray(shap_values[-1])
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3:
            per_row_shap = arr[:, :, -1] if arr.shape[-1] in (1, 2) else arr[:, -1, :]
        else:
            per_row_shap = arr

    assert per_row_shap.shape == (len(sample_idx), len(feature_names)), (
        f"[CHECK FAILED] Per-row SHAP shape {per_row_shap.shape} does not match "
        f"(sample_size={len(sample_idx)}, feature_count={len(feature_names)})."
    )

    for local_i, global_row in enumerate(sample_idx):
        row_shap = per_row_shap[local_i]
        row_nonzero_mask = X_sample_dense_for_masking[local_i] != 0.0
        masked_shap = np.where(row_nonzero_mask, np.abs(row_shap), -np.inf)
        if not row_nonzero_mask.any():
            reason_codes_by_row[global_row] = []
            continue
        top_k = min(N_REASON_CODES, int(row_nonzero_mask.sum()))
        top_feat_idx = np.argsort(masked_shap)[::-1][:top_k]
        codes = [str(feature_names[j]) for j in top_feat_idx]
        for j in top_feat_idx:
            if not row_nonzero_mask[j]:
                grounding_failures += 1
        reason_codes_by_row[global_row] = codes

    all_codes = [c for codes in reason_codes_by_row.values() for c in codes]
    gate5_top_terms = pd.Series(all_codes).value_counts().head(10).index.tolist() if all_codes else []
    print(
        f"[RESULT] Gate 5's own independently-aggregated top reason-code terms (by frequency, "
        f"sampled): {gate5_top_terms}"
    )

except Exception as e:  # noqa: BLE001 - continue gracefully; decision records still get written
    shap_error = f"{type(e).__name__}: {e}"
    gate5_top_terms = []
    print(
        f"[LIMITATION] Per-instance SHAP failed for champion model family "
        f"'{type(CANDIDATES[CHAMPION_NAME]).__name__}': {shap_error}. Decision records below will "
        "still be written with predicted labels/probabilities, but with empty reason_codes."
    )

# ============================================================
# SECTION 12: Cross-check against Gate 4's already-saved global top-10 SHAP features
# ============================================================
gate4_shap_csv_path = ARTIFACTS_DIR / "gate4_shap_top_features.csv"
gate4_top_terms = []
if gate4_shap_csv_path.exists():
    gate4_shap_df = pd.read_csv(gate4_shap_csv_path)
    if len(gate4_shap_df) > 0:
        gate4_top_terms = gate4_shap_df["feature"].head(10).tolist()
overlap_terms = sorted(set(gate5_top_terms) & set(gate4_top_terms))
print(
    f"\n[CHECK] Gate 4 global top-10 vs Gate 5 sample-aggregated top-10 overlap: "
    f"{len(overlap_terms)} terms in common ({overlap_terms})."
)

# ============================================================
# SECTION 13: Assemble decision records for the FULL held-out test set (one row per real test
# example) - includes tags_group (Design decision 2) so a reviewer can filter by it.
# ============================================================
in_sample_set = set(int(i) for i in sample_idx)
records = []
for i in range(len(X_test_raw)):
    codes = reason_codes_by_row.get(i, None) if i in in_sample_set else None
    row_summary = "; ".join(
        f"{col}={X_test_raw.iloc[i][col]}" for col in FEATURE_COLS_CATEGORICAL + [COMPANY_COL]
    )
    records.append(
        {
            "row_index": i,
            "true_label": int(y_test[i]),
            "predicted_label": int(y_pred_default[i]),
            "predicted_probability": round(float(y_proba[i]), 4),
            "correct": bool(y_pred_default[i] == y_test[i]),
            "predicted_label_at_best_f1_threshold": int(y_pred_alt_threshold[i]),
            "tags_group": str(tags_test[i]),
            "in_shap_sample": i in in_sample_set,
            "reason_codes": "|".join(codes) if codes else "",
            "feature_summary": row_summary,
        }
    )
decision_records_df = pd.DataFrame(records)
assert len(decision_records_df) == len(X_test_raw), (
    f"[CHECK FAILED] Decision-record count ({len(decision_records_df)}) does not match test-set "
    f"size ({len(X_test_raw)})."
)

records_path = ARTIFACTS_DIR / "gate5_decision_records.csv"
decision_records_df.to_csv(records_path, index=False)
print(
    f"\n[SAVED] {records_path.relative_to(PROJECT_ROOT)} ({len(decision_records_df):,} decision "
    f"records, {len(in_sample_set):,} with reason codes)"
)

# ============================================================
# SECTION 14: Write summary (idempotent overwrite-in-place) - the recomputed, cross-checked
# disparate-impact finding is surfaced prominently here, not buried.
# ============================================================
summary = {
    "bp_id": "bp3",
    "gate": 5,
    "champion_model": CHAMPION_NAME,
    "n_decision_records": int(len(decision_records_df)),
    "n_with_reason_codes": int(len(in_sample_set)),
    "shap_sample_size_bound": SHAP_SAMPLE_SIZE,
    "n_reason_codes_per_record": N_REASON_CODES,
    "shap_error": shap_error,
    "held_out_test_pr_auc_recomputed": round(recomputed_test_pr_auc, 6),
    "gate3_recorded_held_out_test_pr_auc": round(gate3_recorded_test_pr_auc, 6),
    "pr_auc_consistency_diff": round(pr_auc_consistency_diff, 6),
    "primary_decision_threshold": 0.5,
    "alternate_reference_threshold_from_gate4": gate4_best_f1_threshold,
    "alternate_threshold_note": (
        "predicted_label_at_best_f1_threshold is Gate 4's F1-maximizing threshold from its "
        "9-point grid, carried as an explicitly labeled alternate reference point only - it does "
        "not replace predicted_label, which stays at the default 0.5 threshold Gate 3 officially "
        "reports at."
    ),
    "mean_predicted_probability_correct_predictions": (
        round(mean_conf_correct, 4) if mean_conf_correct is not None else None
    ),
    "mean_predicted_probability_incorrect_predictions": (
        round(mean_conf_incorrect, 4) if mean_conf_incorrect is not None else None
    ),
    "disparate_impact_check": {
        "tags_group_breakdown": disparate_rows,
        "adverse_impact_ratio_recomputed": (
            round(recomputed_adverse_impact_ratio, 4) if recomputed_adverse_impact_ratio is not None else None
        ),
        "gate4_recorded_adverse_impact_ratio": gate4_recorded_adverse_impact_ratio,
        "consistency_diff_vs_gate4": (
            round(adverse_impact_consistency_diff, 6) if adverse_impact_consistency_diff is not None else None
        ),
        "flagged": (
            bool(recomputed_adverse_impact_ratio < 0.8)
            if recomputed_adverse_impact_ratio is not None
            else None
        ),
        "limitation": (
            "Monitoring signal for a human reviewer, not a legal determination of ECOA/Reg B "
            "compliance. Carried forward from Gate 4's finding, independently recomputed here on "
            "this gate's own full refit and full held-out test set."
        ),
    },
    "gate5_aggregated_top_reason_code_terms": gate5_top_terms,
    "gate4_global_top10_terms": gate4_top_terms,
    "overlap_terms_with_gate4": overlap_terms,
    "overlap_count_with_gate4": len(overlap_terms),
    "reason_code_grounding_failures": int(grounding_failures),
    "reason_code_grounding_method": (
        "A reason code is only ever reported for a row if that feature's value is nonzero in "
        "that row - i.e. the category is literally present in that row by construction of the "
        "one-hot encoding, not asserted after the fact."
    ),
    "compliance_touchpoint": {
        "udaap_language_review": (
            "Not Applicable to BP3 Gate 5 - this gate generates no GenAI or customer-facing text; "
            "predicted labels, probabilities, and per-instance SHAP reason codes are deterministic "
            "outputs of the champion classifier. Real GenAI-drafted customer-facing text (subject "
            "to UDAAP review) is scoped to BP6 (GenAI Resolution Assistant) per the Master "
            "Execution Plan, same standing scope decision as BP1/BP2 Gate 5."
        ),
        "nist_ai_rmf_measure_manage": (
            "Not Applicable to BP3 Gate 5 for the same reason - no GenAI output is produced here. "
            "Applies at BP6."
        ),
        "ecoa_reg_b_disparate_impact_monitoring": (
            "Applicable and carried forward from Gate 4 (see disparate_impact_check above) - "
            "recomputed independently here, not simply copied."
        ),
        "genai_api_used": False,
        "scope_decision_confirmed_by_user_utc": "2026-09-22",
    },
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
summary_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"[SAVED] {summary_path.relative_to(PROJECT_ROOT)}")

inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
if inventory_path.exists():
    with open(inventory_path, "r", encoding="utf-8") as f:
        model_inventory_entry = json.load(f)
else:
    model_inventory_entry = {"bp_id": "bp3", "model_name": CHAMPION_NAME}
model_inventory_entry["status"] = "Gate 5 decision layer + reporting complete"
model_inventory_entry["gate5_n_decision_records"] = int(len(decision_records_df))
model_inventory_entry["gate5_held_out_test_pr_auc_recomputed"] = round(recomputed_test_pr_auc, 6)
model_inventory_entry["gate5_adverse_impact_ratio_recomputed"] = (
    round(recomputed_adverse_impact_ratio, 4) if recomputed_adverse_impact_ratio is not None else None
)
model_inventory_entry["gate5_genai_api_used"] = False
model_inventory_entry["gate5_generated_at_utc"] = datetime.now(timezone.utc).isoformat()
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (Gate 5 fields added)")

from utils.bp1_config_sync import write_gate_block  # noqa: E402

status_text = BP3_CONFIG_PATH.read_text(encoding="utf-8")
current_status_line = [ln for ln in status_text.splitlines() if ln.startswith("status:")][0]
new_status_value = (
    current_status_line.split('"')[1] + "_gate5_confirmed"
    if "_gate5_confirmed" not in current_status_line
    else current_status_line.split('"')[1]
)
status_text = re.sub(
    r"^status:.*$", f'status: "{new_status_value}"', status_text, count=1, flags=re.MULTILINE
)
BP3_CONFIG_PATH.write_text(status_text, encoding="utf-8")

gate5_marker = "# --- Gate 5 (Decision Layer & Reporting) results (appended, idempotent overwrite) ---"
gate5_block_lines = [
    "gate5_decision_layer:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f"  n_decision_records: {int(len(decision_records_df))}",
    f"  n_with_reason_codes: {int(len(in_sample_set))}",
    f"  held_out_test_pr_auc_recomputed: {round(recomputed_test_pr_auc, 6)}",
    "  adverse_impact_ratio_recomputed: "
    f"{round(recomputed_adverse_impact_ratio, 4) if recomputed_adverse_impact_ratio is not None else 'null'}",
    "  genai_api_used: false",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(BP3_CONFIG_PATH, gate5_marker, gate5_block_lines)
print(f"[SAVED] {BP3_CONFIG_PATH.relative_to(PROJECT_ROOT)} (gate5_decision_layer block)")

# ============================================================
# SECTION 15: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_matches_gate3_and_gate4_recorded": CHAMPION_NAME == gate3_block["champion_model"],
    "decision_records_count_equals_test_set_size": len(decision_records_df) == len(X_test_raw),
    "all_records_have_predicted_label_and_probability": decision_records_df["predicted_label"].notna().all()
    and decision_records_df["predicted_probability"].notna().all(),
    "predicted_probability_within_valid_range": decision_records_df["predicted_probability"]
    .between(0.0, 1.0)
    .all(),
    "positive_class_probabilities_within_valid_range": proba_valid_range,
    "shap_sample_size_matches_configured_bound": len(sample_idx) == SHAP_SAMPLE_SIZE
    or shap_error is not None,
    "all_reason_codes_grounded_by_nonzero_value": grounding_failures == 0,
    "pr_auc_consistency_with_gate3_recorded": pr_auc_consistency_diff < 1e-4,
    "disparate_impact_consistency_with_gate4_recorded": (
        adverse_impact_consistency_diff is None or adverse_impact_consistency_diff < 1e-4
    ),
    "tags_never_used_as_model_feature": "Tags" not in FEATURE_COLS_CATEGORICAL
    and "Tags" not in [COMPANY_COL],
    "no_barred_column_in_feature_frame": all(
        b not in (FEATURE_COLS_CATEGORICAL + [COMPANY_COL]) for b in BARRED_COLUMNS
    ),
    "no_accuracy_metric_computed_anywhere": True,  # by construction - never called in this notebook
    "compliance_touchpoint_documented": "compliance_touchpoint" in summary
    and bool(summary["compliance_touchpoint"]),
    "disparate_impact_check_documented": "disparate_impact_check" in summary
    and bool(summary["disparate_impact_check"]),
    "decision_records_csv_written": records_path.exists(),
    "summary_json_written": summary_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp3_config_yaml_updated": BP3_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP3 Gate 5 complete. {len(decision_records_df):,} decision records "
    f"written ({len(in_sample_set):,} with grounded reason codes). Recomputed held-out test "
    f"PR-AUC={round(recomputed_test_pr_auc, 4)} (Gate 3 recorded: "
    f"{round(gate3_recorded_test_pr_auc, 4)}). Recomputed disparate-impact ratio (Tags, "
    f"monitoring only)="
    f"{round(recomputed_adverse_impact_ratio, 4) if recomputed_adverse_impact_ratio is not None else 'n/a'} "
    f"(Gate 4 recorded: {gate4_recorded_adverse_impact_ratio}). Gate 4/Gate 5 SHAP top-term "
    f"overlap: {len(overlap_terms)}/10. No GenAI API used (offline decision-record layer, same "
    "scope decision as BP1/BP2). Proceed to BP3 Gate 6 (Productization, Monitoring & Governance) "
    "next."
)
